# Using Picaso for Roman Coronagraph Exoplanet Atmospheric Modeling

In this hands-on tutorial you will learn the basic picaso procedures that can launch you into assessing a target’s observability and science prospects given Roman-CGI's specifications. You will also be equipped to move on to the provided quickstart guides for climate calculations, grid-fitting, and retrievals. 

We'll cover:
* Using picaso to calculate optical spectra of substellar companions with both reflected light and thermal emission components
* Familiarity with the physical information encoded in these observations and in which regions of parameter space reflected light vs self-luminosity dominates
* Familiarity with some relevant self-consistent 1d-RCE model grids for Roman-CGI targets
* Simple methods for simulating noisy Roman-CGI observations and chi^2 grid fitting

It is expected that you attended the accompanying conceptual lecture given by Zarah and are comfortable working in python. Notebooks A-C are designed to take ~3 hours for a brand new picaso user to work through. If you finish early, you are invited to apply your new skills towards a self-directed exploration of Roman-CGI science potential, or engaging with one of the quickstart guides.

jump down to [colab set up](#colab) and [local installation](#local)

------------------------
## Roman-CGI Overview 

Roman-CGI is a NASA technology demonstrator flying with the flagship Roman Space Telescope. For the first time in space it will use deformable mirrors and wave-front sensing and control to serve as a key stepping stone towards HWO. Roman will observe at shorter wavelengths than we've been able to directly image any exoplanets before this.

**Roman-CGI Observing Modes:**

| Band |Mode |  Center | Width | Resolution | Data points |
|------|------|--------|--------|------------|-------------|
|  Band 1 |Photometric  | 573.8 nm | 9.8% | -- | 1 |
|  Band 3 |Spectroscopic |729.3 nm | 16.8% | R = 50 | ~8 |
|  Band 4 |Photometric  | 825.5 nm | 11.7%  |  -- | 1|

As part of its tech demo, Technology Threshold Requirement 5 (**TTR5**) states that Roman-CGI will make a measurement with SNR > 5 at a separation of 6-9 $\lambda$/D, with a contrast of 10^-7 in a 10% bandwidth at a central wavelength < 600 nm. The contrast and working angle relative to other facilities and possible targets is shown in panel (A) below. 


![alt text](Images/roman_contrast_curves_targets.png)

Roman-CGI will likely demonstrate TTR5 on a **~2400 K brown dwarf** companion to a main-sequence host star, HIP 71618 B. At this temperature, the light collected by Roman-CGI will be dominated by **thermal emission**. Roman-CGI will likely also carry out photometric and spectral observations of some additional young, hot, and widely orbitting substellar companions with a broad range of temperatures ~700-2400 K.

For the first time ever, Roman-CGI will aim to image **1-3 old+cold exoplanets** that reside near their host stars' snow-lines, true solar system gas giant analogues! These have been detected via radial velocity but never imaged. The light collected from them will be totally dominated by **reflected light** from their host star in Roman-CGI's wavelength range. 

The objects currently planned to make up part of Roman-CGI's observing schedule are marked with red outlines and placed in the context of other exoplanets in the NASA exoplanet archive in panel (B above). Some of their properties relevant for atmosphere modeling are summarized in the Table below.

### Table of Approximate Properties for Top Roman-CGI Targets

---

**RV planets (reflected-light targets)**

| Name | Tint (K) | Mass (MJup) | log g (cgs)| Radius (RJup) | a (AU) | Star Teff (K) | Star log g | Star Radius (Rsun) |
|------|------------|----------------|----------|-----------------|--------|------------|---------|-----------------|
| 14 Her b | ~142 | 8.5 ± 1.0 | — | — | 2.77 | 5270 | 4.5 * | 0.87 |
| ups And d | ~238 | 10.25 | — | — | 2.51 | 6160 | 4.1 * | 1.62 |
| eps Eri b | ~112 | 0.66 | — | — | 3.53 | 5084 | 4.6 * | 0.735 |
| 47 Uma c | ~165 | ≥0.54 † | — | — | 3.60 | 5872 | 4.30 | 1.21 |

---

We don't have radii and surface gravities for the RV planets yet! Something like https://github.com/chenjj2/forecaster could be a good way to pick something reasonable based on their radii.

**Directly imaged self-luminous companions**

| Name | Tint (K) | Mass (MJup) | log g (cgs) | Radius (RJup) | a (AU) | Star Teff (K) | Star log g | Star Radius (Rsun) |
|------|------------|----------------|----------|-----------------|--------|------------|---------|-----------------|
| HIP 71618 B | 2700 ± 100 | 60 (+27/−21) | 4.5 | 1.70 | 11.1 (+1.3/−1.0) | 9300 (+400/−500) | 4.3 * | ~2.0 * |
| HIP 54515 b | ~2000 | 17.7 (+7.6/−4.9) | — | — | ~25 | ~8200 * | ~4.2 * | ~1.8 * |
| beta Pic b | 1607 (+5/−6) | 9–13 | 4.46 (+0.02/−0.04) | 1.46 | ~9.2 | 8054 | 4.18 | 1.53 |
| kappa And b | 1791 ± 68 | 17.3 ± 1.8 | 4.35 ± 0.07 | 1.42 ± 0.06 | 55 | 10900 | 3.78 | — |
| HR 8799 c | ~1150 | ~9–10 | ~3.5–4.0 | ~1.2 | 38 | 7400 | 4.0 | ~1.44 * |
| HR 8799 d | ~1100 | ~9–10 | ~3.5–4.0 | ~1.2 | 24 | ″ | ″ | ″ |
| HR 8799 e | ~1200 | ~10 | ~3.5–4.0 | ~1.2 | 16.4 | ″ | ″ | ″ |


**CAUTION:** These quantities include many rough estimates for your pedagogical purposes, not publication ready!

------------------------
## picaso Overview


![alt text](Images/phase_angle_disco.png)

- started as the first open-source reflected light post-processing tool
- grew out of legacy fortran codes, upgraded to object-oriented and modular pythonic design
- extensive additional tutorials and documentation available at: https://natashabatalha.github.io/picaso/tutorials.html#
- now includes most capabilities needed for modeling exoplanets or brown dwarfs:
  - reflected light spectra
  - thermal emission spectra
  - transit spectra
  - phase curves with 3d variation around planet
  - 1D RCE climate modeling, including irradiation, clouds, disequilibrium chemistry, photochemistry
  - companion virga package handles cloud calculations
  - built-in tools for grid-fitting
  - retrieval framework (coming soon)
  - GPU version of the code (coming soon)
- now the main code base for producing Sonora family of grid models, paired with evolutionary code as boundary conditions to get cooling curves
- latest picaso version release was last week! version 4: https://arxiv.org/abs/2602.22468
- ethos of open source reproducible science


------------------------
## <a id="colab"></a> Instructions for Working in google colab 
- advantage here is keeping things cleanly separated from your local set up, uniform operating system for all of us
- since picaso and virga are fully python they tend to work ok on most people's machines
- WIP... will have something similar to Sagan School up ASAP https://nexsci.caltech.edu/workshop/2023/handson.shtml 

--------------------
## <a id="local"></a> (optional) Local "developer" installation of picaso and virga for more extensive use

#### First make a new virtual environment and activate it

`$ conda create -n picaso_v4 python=3.12`

`$ conda activate picaso_v4`

#### I like to also install my preferred code development environment

`$ conda install -c conda-forge jupyterlab`

#### Install the virga python package which handles clouds for picaso

`$ git clone https://github.com/natashabatalha/virga.git` 

`$ cd virga`

`$ pip install -e .`

`$ cd ../`

#### Depending on which order you install virga and picaso, you can end up with some version incompatabilities of pandas and numpy, so let's make sure they are both set to versions that work with picaso

`$ conda run -n picaso_v4 pip install "pandas==2.3.0"`

`$ conda run -n picaso_v4 pip install "numpy==1.26.4"`

#### Install the picaso python package, checking out the dev branch so we can use version 4

`$ git clone https://github.com/natashabatalha/picaso.git`

`$ cd picaso`

`$ git checkout dev`

`$ pip install -e .`

#### Last set-up step is downloading the reference data, we'll do this within python using picaso's interactive data download module

In [1]:
# you need to set environment variables called 
# picaso_refdata and PYSYN_CDBS 
# every time you use picaso
import os
# ── SET YOUR PATHS HERE ──────────────────────────────────────────
PICASO_REFDATA = '/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/reference'
PYSYN_CDBS = '/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/reference/stellar_grids'
# ─────────────────────────────────────────────────────────────────
os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/reference


In [ ]:
# now you can start to download some additional resources from zenodo
# using convenient routines in picaso.data module
import picaso.data as data

In [ ]:
# main medium resolution opacity file, this fills 6.04 GB
data.get_data(category_download='resampled_opacity',target_download='default')

In [ ]:
# stellar models 
data.get_data(category_download='stellar_grids',target_download='ck04models')

In [ ]:
# pre-weighted correlated-k tables for doing climate calculations with
# faster lower resolution opacities, this fills 2.4 GB
data.get_data(category_download='ck_tables',target_download='pre-weighted')

In [ ]:
# sonora bobcat TP profiles
data.get_data(category_download='sonora_grids',target_download='bobcat')

In [ ]:
# Mie coeffients for cloud calculations
data.get_data(category_download='virga_mieff',target='default')

In [ ]:
# finally check whether everything worked!
data.check_environ()

----------------------
#### Now you are ready to go! Remember that you'll need the following few lines at the start of your script or notebook every time you want to use picaso, edited to the appropriate file paths for your own computer.

In [1]:
import os
# ── SET YOUR PATHS HERE ──────────────────────────────────────────
PICASO_REFDATA = '/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/reference'
PYSYN_CDBS = '/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/reference/stellar_grids'
# ─────────────────────────────────────────────────────────────────
os.environ['picaso_refdata'] = PICASO_REFDATA
os.environ['PYSYN_CDBS'] = PYSYN_CDBS

from picaso import justdoit as jdi 

/Users/briannalacy/Research/Picaso_Tutorial_Dev/picaso/picaso/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import declare_namespace
